In [ ]:
import os, shutil, subprocess, sys, zipfile
from pathlib import Path

inputs = Path("/kaggle/input")
roots = list((inputs / "nanowm-code").rglob("pyproject.toml"))
if not roots:
    roots = list(inputs.rglob("pyproject.toml"))
if len(roots) != 1:
    raise RuntimeError(f"Expected exactly one project root, found {len(roots)}: {roots}")
mounted = roots[0].parent
root = Path("/kaggle/working/project")
root.mkdir(parents=True, exist_ok=True)
shutil.copytree(mounted, root, dirs_exist_ok=True)
for archive in mounted.glob("*.zip"):
    with zipfile.ZipFile(archive) as handle:
        handle.extractall(root)
os.chdir(root)
sys.path.insert(0, str(root))
print("project root:", root)

In [ ]:
subprocess.run([sys.executable, "scripts/preflight_gpu.py"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "diffusers", "lpips"], check=True)
subprocess.run([sys.executable, "-m", "pytest", "tests/", "-q"], check=True)

In [ ]:
# Inference-side: costs Kaggle weekly quota, not the 10-hour training
# budget. Same M1 trajectory as the smoke run.
subprocess.run([
    sys.executable, "scripts/preprocess/precompute_dataset.py",
    "--tier", "m1",
    "--out-dir", "/kaggle/working/nanowm_data/m1",
    "--device", "cuda",
], check=True)

In [ ]:
# The real M1 training ticket: 0.75 GPU-hours, nearly all of M1's
# remaining 0.789h allocation (0.211h already spent on the pipeline-
# verification smoke attempts). Exit code 2 = ticket deadline reached,
# which is the budget rule working as designed, not a failure.
result = subprocess.run([
    sys.executable, "scripts/train.py",
    "--config", "configs/m1_overfit_5m_full.yaml",
])
print(f"train.py exit code: {result.returncode}")
print(Path("budget/ledger.jsonl").read_text())

In [ ]:
# Same session, so the checkpoint just produced is already on disk --
# no need for the separate assets-dataset round trip the smoke run's gate
# eval needed.
subprocess.run([
    sys.executable, "scripts/run_m1_gate.py",
    "--checkpoint-dir", "/kaggle/working/runs/m1_overfit_5m_full/checkpoints",
    "--data-dir", "/kaggle/working/nanowm_data/m1",
    "--out", "/kaggle/working/m1_gate_full.json",
], check=True)

In [ ]:
import json
print(json.dumps(json.load(open("/kaggle/working/m1_gate_full.json")), indent=2))